In [1]:
import os
print(os.getcwd())

C:\Users\hp\Desktop\Ecommerce-Sales-Analytics\notebooks


In [3]:
import pandas as pd

# Load your rebuilt master dataset
df_master = pd.read_csv('../data/processed/olist_master_v1.csv')

# Filter to only reviews that have written comments (not just a star rating)
reviews_with_text = df_master[df_master['review_comment_message'].notna()].copy()

# Drop duplicates since master is at item-level (an order can appear multiple times)
reviews_with_text = reviews_with_text.drop_duplicates(subset='order_id')[
    ['order_id', 'review_score', 'review_comment_message', 'delivery_delay_days']
]

print(f"Total reviews with text: {reviews_with_text.shape[0]}")
reviews_with_text.head()

Total reviews with text: 38962


,order_id,review_score,review_comment_message,delivery_delay_days
0,e481f51cbdc54678b7cc49136f2d6af7,4.0,"Não testei o produto ainda, mas ele veio corre...",-8.0
3,53cdb2fc8bc7dce0b6741e2150273451,4.0,Muito bom o produto.,-6.0
5,949d5b44dbf5de918fe9c16f97b45f8a,5.0,O produto foi exatamente o que eu esperava e e...,-13.0
12,e6ce16cb79ec1d90b1da9085a6118aeb,1.0,Aguardando retorno da loja,-9.0
17,432aaf21d85167c2c86ec9448c4e42cc,4.0,Gostei do produto,-9.0


In [6]:
from transformers import pipeline

# This model uses standard BERT tokenization (no SentencePiece/tiktoken issues)
# It supports Portuguese and other languages, and outputs 1-5 star ratings
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

# Quick test on one Portuguese review to confirm it works correctly
test_review = reviews_with_text['review_comment_message'].iloc[0]
print("Review:", test_review)
print("Result:", sentiment_pipeline(test_review))

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

C:\Users\hp\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\models--nlptown--bert-base-multilingual-uncased-sentiment. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  669MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Review: Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danificada, o que ficará chato, pois se trata de um presente.
Result: [{'label': '3 stars', 'score': 0.5271695852279663}]


In [7]:
def get_sentiment_label(result):
    """
    Converts the model's '1 star' to '5 stars' output into simple sentiment categories.
    1-2 stars = Negative, 3 stars = Neutral, 4-5 stars = Positive
    """
    stars = int(result[0]['label'][0])  # Extract the number from '3 stars' -> 3
    if stars <= 2:
        return 'Negative'
    elif stars == 3:
        return 'Neutral'
    else:
        return 'Positive'

# Test it on the same review
print(get_sentiment_label(sentiment_pipeline(test_review)))

Neutral


In [8]:
# Sample 3000 reviews randomly (random_state makes it reproducible)
sample_reviews = reviews_with_text.sample(n=3000, random_state=42).reset_index(drop=True)

print(f"Sample size: {sample_reviews.shape[0]}")

Sample size: 3000


In [9]:
from tqdm import tqdm  # progress bar so you can see it's working, not frozen
tqdm.pandas()

# Apply the sentiment pipeline to every review in the sample, showing a progress bar
sample_reviews['sentiment_result'] = sample_reviews['review_comment_message'].progress_apply(
    lambda x: sentiment_pipeline(x[:512])  # truncate to 512 chars, the model's max input length
)

# Extract just the simple sentiment label using our helper function
sample_reviews['sentiment'] = sample_reviews['sentiment_result'].apply(get_sentiment_label)

print(sample_reviews['sentiment'].value_counts())

100%|██████████| 3000/3000 [14:02<00:00,  3.56it/s]

sentiment
Positive    1673
Negative    1047
Neutral      280
Name: count, dtype: int64


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

C:\Users\hp\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

{'sequence': 'Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danificada, o que ficará chato, pois se trata de um presente.', 'labels': ['product quality', 'good experience', 'wrong item received', 'late delivery', 'packaging damage', 'customer service'], 'scores': [0.4285840094089508, 0.34657490253448486, 0.178432434797287, 0.021708926185965538, 0.016704587265849113, 0.0079951835796237]}


In [12]:
from transformers import pipeline

# distilbart is a distilled (smaller, faster) version of bart-large-mnli
# Roughly 2-3x faster with a modest accuracy tradeoff - good fit for this use case
theme_classifier = pipeline(
    "zero-shot-classification",
    model="valhalla/distilbart-mnli-12-3"
)

# Quick test to confirm it works
test_result = theme_classifier(test_review, candidate_themes)
print(test_result)

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

C:\Users\hp\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\models--valhalla--distilbart-mnli-12-3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.02GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

...

model.safetensors: downloading bytes:           |  0.00B            

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

{'sequence': 'Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danificada, o que ficará chato, pois se trata de um presente.', 'labels': ['good experience', 'product quality', 'packaging damage', 'wrong item received', 'customer service', 'late delivery'], 'scores': [0.4824172556400299, 0.3880171775817871, 0.05339720845222473, 0.03689958155155182, 0.023611610755324364, 0.01565719023346901]}


In [13]:
# Reduce to 1000 reviews - still a strong, defensible number, much faster to process
sample_reviews_small = sample_reviews.sample(n=1000, random_state=42).reset_index(drop=True)
print(sample_reviews_small.shape)

(1000, 6)


In [14]:
from tqdm import tqdm
tqdm.pandas()

def get_top_theme(text):
    result = theme_classifier(text[:512], candidate_themes)
    return result['labels'][0]

sample_reviews_small['theme'] = sample_reviews_small['review_comment_message'].progress_apply(get_top_theme)

print(sample_reviews_small['theme'].value_counts())



  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 2/1000 [00:02<24:50,  1.49s/it]

  0%|          | 3/1000 [00:06<38:34,  2.32s/it]

  0%|          | 4/1000 [00:09<45:57,  2.77s/it]

  0%|          | 5/1000 [00:13<48:25,  2.92s/it]

  1%|          | 6/1000 [00:17<54:38,  3.30s/it]

  1%|          | 7/1000 [00:20<56:16,  3.40s/it]

  1%|          | 8/1000 [00:27<1:11:07,  4.30s/it]

  1%|          | 9/1000 [00:34<1:28:09,  5.34s/it]

  1%|          | 10/1000 [00:40<1:28:08,  5.34s/it]

  1%|          | 11/1000 [00:44<1:25:27,  5.18s/it]

  1%|          | 12/1000 [00:49<1:20:46,  4.91s/it]

  1%|▏         | 13/1000 [00:52<1:10:13,  4.27s/it]

  1%|▏         | 14/1000 [00:54<1:01:52,  3.77s/it]

  2%|▏         | 15/1000 [01:03<1:26:39,  5.28s/it]

  2%|▏         | 16/1000 [01:06<1:16:38,  4.67s/it]

  2%|▏         | 17/1000 [01:13<1:28:01,  5.37s/it]

  2%|▏         | 18/1000 [01:16<1:16:03,  4.65s/it]

  2%|▏         | 19/1000 [01:22<1:19:38,  4.87s/it]

  2%|▏         | 20/10

KeyboardInterrupt: 

In [15]:
import re

# Define keyword patterns for each theme, in Portuguese
theme_keywords = {
    'late delivery': ['atraso', 'atrasad', 'demorou', 'demora', 'chegou tarde', 'não chegou'],
    'wrong item received': ['produto errado', 'veio errado', 'diferente do', 'não é o que'],
    'packaging damage': ['amassad', 'danificad', 'quebrad', 'caixa', 'embalagem'],
    'product quality': ['qualidade', 'defeito', 'não funciona', 'ruim', 'péssimo'],
    'customer service': ['atendimento', 'suporte', 'resposta', 'contato'],
    'good experience': ['ótimo', 'excelente', 'recomendo', 'adorei', 'perfeito', 'gostei']
}

def classify_theme_keywords(text):
    """
    Checks review text for keyword matches per theme.
    Returns the theme with the most keyword matches, or 'other' if none match.
    """
    text_lower = text.lower()
    theme_scores = {}
    
    for theme, keywords in theme_keywords.items():
        score = sum(1 for kw in keywords if kw in text_lower)
        if score > 0:
            theme_scores[theme] = score
    
    if not theme_scores:
        return 'other'
    
    # Return the theme with the highest keyword match count
    return max(theme_scores, key=theme_scores.get)

# Test it on the same review as before
print(classify_theme_keywords(test_review))

packaging damage


In [16]:
sample_reviews['theme'] = sample_reviews['review_comment_message'].apply(classify_theme_keywords)

print(sample_reviews['theme'].value_counts())

theme
other                  1789
good experience         672
product quality         236
late delivery           118
packaging damage         82
customer service         75
wrong item received      28
Name: count, dtype: int64


In [17]:
negative_themes = sample_reviews[sample_reviews['sentiment'] == 'Negative']['theme'].value_counts()
print("Top themes in NEGATIVE reviews:")
print(negative_themes)

print("\nTop themes overall:")
print(sample_reviews['theme'].value_counts())

Top themes in NEGATIVE reviews:
theme
other                  718
late delivery           80
product quality         66
good experience         63
packaging damage        61
customer service        37
wrong item received     22
Name: count, dtype: int64

Top themes overall:
theme
other                  1789
good experience         672
product quality         236
late delivery           118
packaging damage         82
customer service         75
wrong item received      28
Name: count, dtype: int64


In [18]:
sample_reviews['theme'] = sample_reviews['review_comment_message'].apply(classify_theme_keywords)

print(sample_reviews['theme'].value_counts())

theme
other                  1789
good experience         672
product quality         236
late delivery           118
packaging damage         82
customer service         75
wrong item received      28
Name: count, dtype: int64
